# Random Forest con presencias y pseudoausencias GEE

**Objetivo:** RF con presencias y pseudoausencias extraídas de GEE; 9 variables.

Este cuaderno contiene el código completo del experimento `train_final_model.py`,
dividido en etapas y acompañado de explicaciones. No necesita ZIP, clonar un
repositorio ni ejecutar un script oculto. Ejecuta las celdas en orden.

**Recorrido:** entorno → carga e inspección → preparación → modelo o análisis →
evaluación → gráficos → descarga y colaboración.

**Entradas:** `incendios_ecuador_variables_extraidas.csv`, `dataset_ausencias_ecuador.csv`.

Los datos no se incluyen. Se conservan los valores predeterminados del protocolo histórico.
Las salidas se generarán al ejecutar; las figuras históricas no están incrustadas.


## 1. Entorno y parámetros
Modifica los parámetros aquí para que una colaboración sea fácil de revisar. Los valores originales siguen seleccionados. Cada ejecución crea una carpeta nueva.

In [ ]:
# Instala únicamente las bibliotecas que falten en el entorno.
import importlib.util
import importlib.metadata
import subprocess
import sys
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image

DEPENDENCIAS = {'numpy': 'numpy', 'pandas': 'pandas', 'sklearn': 'scikit-learn'}
faltantes = [paquete for modulo, paquete in DEPENDENCIAS.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *faltantes], check=True)

CONFIG = dict(seed=42, rf_trees=100, rf_depth=15, epochs=50,
              learning_rate=0.002, batch_size=256)
EXPERIMENTO = 'rf_gee'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
DATA_DIR = Path(globals().get('DATA_DIR_OVERRIDE', 'data')).resolve()
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR_OVERRIDE', Path('results') / EXPERIMENTO / RUN_ID)).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLBACKEND', 'Agg')
print('Parámetros:', CONFIG)
print('Salidas:', OUTPUT_DIR)

def mostrar_figura(ruta):
    # Conserva la figura en disco y también la muestra en el cuaderno.
    if str(ruta).lower().endswith('.png'):
        display(Image(filename=str(ruta)))


## 2. Carga e inspección de los datos
En Colab, la siguiente celda ofrece el selector si faltan archivos. Carga solo los CSV indicados, conservando sus nombres. En un entorno local, colócalos en `data/`. Para el dataset realista usa la copia de la subcarpeta `resultados` original.

In [ ]:
import pandas as pd

ARCHIVOS_REQUERIDOS = ['incendios_ecuador_variables_extraidas.csv', 'dataset_ausencias_ecuador.csv']
def localizar(nombre):
    candidatos = [DATA_DIR / nombre, DATA_DIR / 'resultados' / nombre]
    return next((p for p in candidatos if p.is_file()), candidatos[0])

pendientes = [n for n in ARCHIVOS_REQUERIDOS if not localizar(n).is_file()]
if pendientes:
    print('Archivos requeridos:', pendientes)
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloca los CSV en ' + str(DATA_DIR))
    cargados = files.upload()
    for nombre, contenido in cargados.items():
        if Path(nombre).name != nombre or nombre not in ARCHIVOS_REQUERIDOS:
            raise ValueError('Archivo inesperado: ' + nombre)
        (DATA_DIR / nombre).write_bytes(contenido)
    del cargados

DATA_FILES = {n: localizar(n) for n in ARCHIVOS_REQUERIDOS}
if any(not p.is_file() for p in DATA_FILES.values()):
    raise FileNotFoundError('Faltan CSV; repite esta celda y carga los indicados.')

# Inspección visible: dimensiones, tipos, valores faltantes y balance de clases.
for nombre, ruta in DATA_FILES.items():
    tabla = pd.read_csv(ruta)
    print('\n', nombre, '| filas y columnas:', tabla.shape)
    display(tabla.head())
    display(pd.DataFrame({'tipo': tabla.dtypes.astype(str), 'nulos': tabla.isna().sum()}))
    if 'clase' in tabla.columns:
        display(tabla['clase'].value_counts(dropna=False).rename('filas').to_frame())

REGISTRO = dict(experimento=EXPERIMENTO, parametros=CONFIG, run_id=RUN_ID,
    input_sha256={n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in DATA_FILES.items()},
    versiones={p: importlib.metadata.version(p) for p in DEPENDENCIAS.values()},
    estado='iniciado')
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')


## 3. Bibliotecas, rutas y reproducibilidad
Estas son las importaciones, rutas de entrada/salida y semillas del experimento. Los archivos generados se guardan en la carpeta de esta ejecución.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
BASE_DIR = str(OUTPUT_DIR)
FILE_PRESENCIAS = str(DATA_FILES['incendios_ecuador_variables_extraidas.csv'])
FILE_AUSENCIAS = str(DATA_FILES['dataset_ausencias_ecuador.csv'])
FILE_COMBINADO = str(OUTPUT_DIR / 'dataset_entrenamiento_completo.csv')
FILE_MODELO = str(OUTPUT_DIR / 'modelo_rf_incendios.pkl')


## 4. Desarrollo del experimento
Las siguientes celdas contienen el cuerpo del análisis original, con comentarios y código editable. Puedes inspeccionar las variables entre pasos. Ejecuta nuevamente desde la carga de datos si cambias una transformación previa.

### Etapa 1: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:
print("="*60)
print("  ENTRENAMIENTO FINAL: UNIÓN DE DATOS Y MACHINE LEARNING  ")
print("="*60)

# Verificar existencia de archivos
if not os.path.exists(FILE_PRESENCIAS):
    print(f"[ERROR] No se encontro el archivo de presencias: {FILE_PRESENCIAS}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
if not os.path.exists(FILE_AUSENCIAS):
    print(f"[ERROR] No se encontro el archivo de ausencias: {FILE_AUSENCIAS}")
    raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
# 1. Cargar Presencias (Incendios Reales)
df_presencias = pd.read_csv(FILE_PRESENCIAS)
df_presencias['clase'] = 1
print(f"[OK] Cargadas {len(df_presencias)} presencias (incendios reales) desde GEE.")

# 2. Cargar Ausencias (Zonas sin Incendio reales de GEE)
df_ausencias = pd.read_csv(FILE_AUSENCIAS)
df_ausencias['clase'] = 0
print(f"[OK] Cargadas {len(df_ausencias)} ausencias (zonas de control) desde GEE.")

# Definir variables predictoras
features = ['elevacion', 'pendiente', 'orientacion', 'cobertura_suelo', 'ndvi', 'evi', 'nbr', 'temp_c', 'precip_mm']

# Alinear columnas de ambos datasets antes de concatenar
# Conservamos clase + variables predictoras
cols_to_keep = ['clase'] + features


### Etapa 2: Carga y preparación de datos
Comprueba las columnas, las filas descartadas y el significado de la etiqueta antes de entrenar. El código conserva la selección de variables del experimento original.

In [ ]:

# Nos aseguramos de que existan todas las columnas
for col in cols_to_keep:
    if col not in df_presencias.columns:
        print(f"[ERROR] La columna {col} no esta en el archivo de presencias.")
        raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
    if col not in df_ausencias.columns:
        print(f"[ERROR] La columna {col} no esta en el archivo de ausencias.")
        raise ValueError("Revisa el archivo y las columnas indicadas en la salida anterior.")
df_p_clean = df_presencias[cols_to_keep]
df_a_clean = df_ausencias[cols_to_keep]

# 3. Concatenar bases de datos
df_completo = pd.concat([df_p_clean, df_a_clean], ignore_index=True)

# Limpiar posibles filas con valores nulos
df_completo = df_completo.dropna(subset=features)

# Guardar la base de datos completa unificada por si la necesitas para otro software (R, QGIS, etc.)
df_completo.to_csv(FILE_COMBINADO, index=False)
print(f"[OK] Base de datos unificada guardada en: {FILE_COMBINADO}")

# Separar X (predictores) e y (clase)
X = df_completo[features]
y = df_completo['clase']

print(f"\n[INFO] Tamano total del dataset final: {X.shape[0]} muestras.")


### Etapa 3: Ajuste del modelo
Aquí se aprende a partir de las muestras de entrenamiento. Revisa los hiperparámetros visibles y evita ajustar decisiones usando repetidamente los resultados del conjunto de prueba.

In [ ]:
print(f"   - Clase 1 (Incendios): {sum(y == 1)}")
print(f"   - Clase 0 (Ausencias): {sum(y == 0)}")

# 4. Dividir en Entrenamiento (80%) y Validacion (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)

print("\n[INFO] Entrenando modelo Random Forest definitivo...")
# Crear y entrenar clasificador
model = RandomForestClassifier(
    n_estimators=CONFIG['rf_trees'], 
    max_depth=CONFIG['rf_depth'], 
    random_state=CONFIG['seed'], 
    n_jobs=-1
)
model.fit(X_train, y_train)
print("[OK] Modelo entrenado exitosamente.")

# 5. Evaluacion del Modelo Definitivo
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, y_proba)

print("\n" + "="*40)
print("  METRICAS DEL MODELO DEFINITIVO")


### Etapa 4: Evaluación
Estas métricas pertenecen al protocolo y a la partición de este cuaderno. Revisa precisión, recall y matriz de confusión además del accuracy.

In [ ]:
print("="*40)
print(f"Precision Global (Accuracy): {acc:.4f} ({acc*100:.2f}%)")
print(f"Area Bajo la Curva ROC (AUC): {roc_auc:.4f}")

print("\n[INFO] Reporte de Clasificacion:")
print(classification_report(y_val, y_pred, target_names=['Ausencia', 'Presencia']))

print("[INFO] Matriz de Confusion:")
cm = confusion_matrix(y_val, y_pred)
print(f"   [[Verdaderos Negativos: {cm[0][0]}, Falsos Positivos: {cm[0][1]}]")
print(f"    [Falsos Negativos:      {cm[1][0]}, Verdaderos Positivos: {cm[1][1]}]]")

# 6. Importancia de Variables Real
print("\n" + "="*40)
print("  IMPORTANCIA REAL DE LAS VARIABLES")
print("="*40)
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

for i, idx in enumerate(indices):
    print(f"{i+1}. {features[idx]:<18}: {importances[idx]*100:.2f}%")

# 7. Guardar el modelo entrenado en disco
with open(FILE_MODELO, 'wb') as f:
    pickle.dump(model, f)
print(f"\n[OK] Modelo entrenado guardado en: {FILE_MODELO}")


### Etapa 5: Inspección y cálculo intermedio
Ejecuta esta etapa después de la anterior y revisa los valores impresos. Las variables quedan disponibles para explorarlas o modificarlas en otra celda.

In [ ]:
print("Modelo guardado. La transferencia a otros territorios requiere validación independiente.")
print("="*60)


## 5. Inspeccionar y guardar resultados
Las tablas se muestran debajo. El registro identifica parámetros, versiones y hashes. Esta celda marca el registro como completado: ejecútala solo cuando hayan terminado todas las etapas anteriores.

In [ ]:
for ruta in sorted(OUTPUT_DIR.glob('*.csv')):
    print(ruta.name)
    display(pd.read_csv(ruta).head(20))
REGISTRO['estado'] = 'completado'
(OUTPUT_DIR / 'ejecucion.json').write_text(json.dumps(REGISTRO, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salidas:', [p.name for p in sorted(OUTPUT_DIR.iterdir())])


In [ ]:
# Descarga las salidas antes de cerrar una sesión de Colab.
import shutil
paquete = shutil.make_archive(str(OUTPUT_DIR.parent / (EXPERIMENTO + '-' + RUN_ID)), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
except ImportError:
    print('ZIP guardado en:', paquete)
else:
    files.download(paquete)


## 6. Proponer una colaboración

1. Plantea una pregunta concreta: por ejemplo, el efecto de un hiperparámetro o de una variable.
2. Conserva una ejecución con la configuración original y luego cambia una sola decisión.
3. Registra el hash del dataset, la partición, la semilla y todas las métricas relevantes.
4. En GitHub, abre un Issue para discutir la propuesta. Trabaja en un fork o en una rama
   con permisos y envía un pull request que explique el cambio y su evidencia.
5. Antes del commit, borra las salidas del cuaderno y revisa que no incluyan datos privados.
   Adjunta resultados únicamente cuando tengas permiso para compartirlos.

Edita la siguiente celda Markdown para documentar tu propuesta. No se envía nada automáticamente.

### Registro de la propuesta

- **Autor y objetivo:**
- **Hipótesis:**
- **Parámetro o código modificado:**
- **Dataset y hash:**
- **Semilla y partición:**
- **Resultado original y resultado de la variante:**
- **Limitaciones y conclusión:**
